<a href="https://colab.research.google.com/github/eyobedb/Multimodal-papaya-disease-classification-Leveraging-Computer-vision-and-NLP/blob/main/Multimodal_VGG19_GPT_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ============================================================
# Multimodal Classification using VGG19 + GPT-2
# Papaya Disease Classification
# ============================================================

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer, GPT2Model
from sklearn.metrics import confusion_matrix, classification_report

# ------------------------------------------------------------
# 1. Device setup
# ------------------------------------------------------------

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ============================================================
# 2. DATASET PATHS
# ============================================================

In [ ]:
train_dir = "/content/dataset/train"
val_dir = "/content/dataset/val"
test_dir = "/content/dataset/test"

# ============================================================
# 3. PARAMETERS
# ============================================================

In [ ]:
IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 1e-4

# ============================================================
# 4. IMAGE TRANSFORMS
# ============================================================

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


# ============================================================
# 5. DATASETS
# ============================================================

In [ ]:
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=test_transform)
test_dataset = datasets.ImageFolder(test_dir, transform=test_transform)

train_loader = DataLoader(train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True)

val_loader = DataLoader(val_dataset,
                        batch_size=BATCH_SIZE,
                        shuffle=False)

test_loader = DataLoader(test_dataset,
                         batch_size=BATCH_SIZE,
                         shuffle=False)

classes = train_dataset.classes
num_classes = len(classes)

print("\nClasses:", classes)

# ============================================================
# 6. CLASS DESCRIPTIONS
# ============================================================

In [ ]:
class_descriptions = {
    "Black_spot_papaya":
        "Leaves covered with black spots indicating black spot infection.",

    "Powdery_mildew":
        "Papaya leaves showing powdery mildew infection typical of powdery disease.",

    "Ring_spot_papaya":
        "Ring spot disease on papaya leaves indicating ring spot infection.",

    "Healthy_papaya":
        "Healthy papaya leaf with no visible disease or damage."
}

# ============================================================
# 7. LOAD GPT-2
# ============================================================

In [ ]:
print("\nLoading GPT-2...")

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

gpt2_model = GPT2Model.from_pretrained("gpt2")
gpt2_model = gpt2_model.to(device)

gpt2_model.eval()

# ============================================================
# 8. CREATE TEXT EMBEDDINGS
# ============================================================

In [ ]:
print("\nGenerating GPT-2 text embeddings...")

text_features = {}

with torch.no_grad():

    for cls_name, description in class_descriptions.items():

        inputs = tokenizer(
            description,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        outputs = gpt2_model(**inputs)

        embedding = outputs.last_hidden_state.mean(dim=1)

        text_features[cls_name] = embedding.squeeze().cpu().numpy()

text_dim = embedding.shape[-1]

print("Text embedding size:", text_dim)

# ============================================================
# 9. GET TEXT FEATURES FOR BATCH
# ============================================================

In [ ]:
def get_text_batch(labels):

    batch_text = []

    for lbl in labels:

        class_name = classes[lbl]

        batch_text.append(text_features[class_name])

    batch_text = np.array(batch_text)

    return torch.tensor(batch_text, dtype=torch.float32)

# ============================================================
# 10. MULTIMODAL MODEL (VGG19 + GPT-2)
# ============================================================

In [ ]:
class VGG19_GPT2(nn.Module):

    def __init__(self, num_classes, text_dim):

        super(VGG19_GPT2, self).__init__()

        # ---------------------------
        # VGG19 IMAGE MODEL
        # ---------------------------

        self.vgg19 = models.vgg19(pretrained=True)

        # Freeze convolution layers
        for param in self.vgg19.features.parameters():
            param.requires_grad = False

        # Replace classifier
        self.vgg19.classifier = nn.Sequential(
            nn.Linear(25088, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(4096, 512),
            nn.ReLU(),
            nn.Dropout(0.5)
        )

        # ---------------------------
        # GPT-2 TEXT BRANCH
        # ---------------------------

        self.text_branch = nn.Sequential(
            nn.Linear(text_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # ---------------------------
        # FUSION LAYERS
        # ---------------------------

        self.fusion = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, num_classes)
        )

    def forward(self, images, text):

        image_features = self.vgg19(images)

        text_features = self.text_branch(text)

        combined = torch.cat((image_features, text_features), dim=1)

        output = self.fusion(combined)

        return output

# ============================================================
# 11. INITIALIZE MODEL
# ============================================================

In [ ]:

model = VGG19_GPT2(num_classes, text_dim)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ============================================================
# 12. TRAINING LOOP
# ============================================================

In [ ]:
print("\nStarting training...\n")

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    loop = tqdm(train_loader)

    for images, labels in loop:

        images = images.to(device)
        labels = labels.to(device)

        text_batch = get_text_batch(labels.cpu().numpy())
        text_batch = text_batch.to(device)

        optimizer.zero_grad()

        outputs = model(images, text_batch)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

        loop.set_description(f"Epoch [{epoch+1}/{EPOCHS}]")
        loop.set_postfix(
            loss=loss.item(),
            accuracy=100 * correct / total
        )

    train_acc = 100 * correct / total

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {running_loss/len(train_loader):.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")

    ========================================================
     VALIDATION
    ========================================================

In [ ]:
model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            text_batch = get_text_batch(labels.cpu().numpy())
            text_batch = text_batch.to(device)

            outputs = model(images, text_batch)

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total

    print(f"Validation Accuracy: {val_acc:.2f}%\n")

# ============================================================
# 13. TESTING
# ============================================================

In [ ]:
print("\nEvaluating on test dataset...\n")

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in tqdm(test_loader):

        images = images.to(device)
        labels = labels.to(device)

        text_batch = get_text_batch(labels.cpu().numpy())
        text_batch = text_batch.to(device)

        outputs = model(images, text_batch)

        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())

        all_labels.extend(labels.cpu().numpy())

# ============================================================
# 14. CONFUSION MATRIX
# ============================================================


In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=classes,
    yticklabels=classes
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - VGG19 + GPT-2")

plt.show()

# ============================================================
# 15. CLASSIFICATION REPORT
# ============================================================

In [ ]:
print("\nClassification Report:\n")

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=classes
    )
)

# ============================================================
# 16. SAVE MODEL
# ============================================================

In [ ]:
torch.save(model.state_dict(), "vgg19_gpt2_multimodal.pth")

print("\nModel saved successfully as:")
print("vgg19_gpt2_multimodal.pth")